In [1]:
from pathlib import Path
import pandas as pd

# Original CSV location
DATA_DIR = Path(r"C:\Users\panda\Downloads\MERGED_CSV\MERGED_CSV")

# Task 2 folder
TASK2_DIR = Path(r"C:\Users\panda\Downloads\GreenFed-IDS\Task2_Preprocessing")

# Output folders
REPORT_DIR = TASK2_DIR / "reports"
OUTPUT_DIR = TASK2_DIR / "output"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FILES = ["Merged01.csv", "Merged02.csv"]

CHUNK_SIZE = 100_000

print("Input folder:", DATA_DIR)
print("Task 2 folder:", TASK2_DIR)

Input folder: C:\Users\panda\Downloads\MERGED_CSV\MERGED_CSV
Task 2 folder: C:\Users\panda\Downloads\GreenFed-IDS\Task2_Preprocessing


In [2]:
# TASK 2 - STEP 1
# Verify dataset structure across ALL chunks

structure_results = []

reference_columns = None
reference_dtypes = None

for filename in FILES:

    path = DATA_DIR / filename

    print("\n" + "=" * 60)
    print("Checking:", filename)

    first_chunk = True
    chunk_count = 0
    column_order_ok = True
    dtype_issues = []

    for chunk in pd.read_csv(
        path,
        chunksize=CHUNK_SIZE,
        low_memory=False
    ):

        chunk_count += 1

        current_columns = list(chunk.columns)

        # --------------------------------
        # Check columns
        # --------------------------------
        if first_chunk:

            file_columns = current_columns

            reference_dtypes = {
                col: str(chunk[col].dtype)
                for col in current_columns
            }

            if reference_columns is None:
                reference_columns = current_columns

            first_chunk = False

        else:

            if current_columns != file_columns:
                column_order_ok = False

        # --------------------------------
        # Check data types
        # --------------------------------
        for col in current_columns:

            current_dtype = str(chunk[col].dtype)

            if current_dtype != reference_dtypes[col]:

                dtype_issues.append({
                    "chunk": chunk_count,
                    "column": col,
                    "expected": reference_dtypes[col],
                    "found": current_dtype
                })

    # --------------------------------
    # Check Label
    # --------------------------------

    label_exists = "Label" in file_columns

    feature_count = (
        len(file_columns) - 1
        if label_exists
        else None
    )

    structure_results.append({
        "file": filename,
        "chunks_checked": chunk_count,
        "column_count": len(file_columns),
        "column_order_matches": column_order_ok,
        "Label_exists": label_exists,
        "candidate_features": feature_count,
        "dtype_consistent_across_chunks": len(dtype_issues) == 0
    })

    print("Chunks checked:", chunk_count)
    print("Columns:", len(file_columns))
    print("Column order OK:", column_order_ok)
    print("Label exists:", label_exists)
    print("Candidate features:", feature_count)
    print("Data types consistent:", len(dtype_issues) == 0)

    if dtype_issues:
        print("\nData type differences:")
        for issue in dtype_issues:
            print(issue)


Checking: Merged01.csv
Chunks checked: 8
Columns: 40
Column order OK: True
Label exists: True
Candidate features: 39
Data types consistent: True

Checking: Merged02.csv
Chunks checked: 8
Columns: 40
Column order OK: True
Label exists: True
Candidate features: 39
Data types consistent: True


In [3]:
# Compare Merged01 and Merged02

columns_01 = None
columns_02 = None

for filename in FILES:

    path = DATA_DIR / filename

    for chunk in pd.read_csv(
        path,
        chunksize=CHUNK_SIZE,
        low_memory=False
    ):
        if filename == "Merged01.csv":
            columns_01 = list(chunk.columns)
        else:
            columns_02 = list(chunk.columns)

        break


print("\n" + "=" * 60)
print("FINAL STRUCTURE COMPARISON")
print("=" * 60)

print("Same column names and order:",
      columns_01 == columns_02)

print("Merged01 column count:",
      len(columns_01))

print("Merged02 column count:",
      len(columns_02))

print("Label in Merged01:",
      "Label" in columns_01)

print("Label in Merged02:",
      "Label" in columns_02)

print("Candidate features in each:",
      len(columns_01) - 1,
      len(columns_02) - 1)


FINAL STRUCTURE COMPARISON
Same column names and order: True
Merged01 column count: 40
Merged02 column count: 40
Label in Merged01: True
Label in Merged02: True
Candidate features in each: 39 39


In [4]:
# TASK 2 - STEP 2
# Investigate missing values without modifying the data

missing_results = []

for filename in FILES:

    path = DATA_DIR / filename

    print(f"\n{'=' * 60}")
    print(f"Missing-value analysis: {filename}")

    total_rows = 0
    missing_counts = {}
    affected_labels = {
        "Std": {},
        "Variance": {}
    }

    for chunk in pd.read_csv(
        path,
        chunksize=CHUNK_SIZE,
        low_memory=False
    ):

        total_rows += len(chunk)

        # Count missing values in every column
        counts = chunk.isna().sum()

        for col, count in counts.items():
            missing_counts[col] = (
                missing_counts.get(col, 0) + int(count)
            )

        # Investigate missing Std and Variance values by label
        for col in ["Std", "Variance"]:

            if col not in chunk.columns:
                continue

            affected = chunk.loc[
                chunk[col].isna(), "Label"
            ]

            label_counts = affected.value_counts(
                dropna=False
            )

            for label, count in label_counts.items():
                label = str(label)

                affected_labels[col][label] = (
                    affected_labels[col].get(label, 0)
                    + int(count)
                )

    print("Total rows:", total_rows)

    print("\nColumns containing missing values:")

    for col, count in missing_counts.items():
        if count > 0:
            print(f"{col}: {count}")

    print("\nMissing values in Std:",
          missing_counts.get("Std", 0))

    print("Missing values in Variance:",
          missing_counts.get("Variance", 0))

    print("\nLabels associated with missing Std values:")
    print(affected_labels["Std"])

    print("\nLabels associated with missing Variance values:")
    print(affected_labels["Variance"])

    for col, count in missing_counts.items():
        missing_results.append({
            "file": filename,
            "column": col,
            "missing_count": count,
            "total_rows": total_rows,
            "missing_percentage": round(
                count / total_rows * 100, 6
            )
        })


Missing-value analysis: Merged01.csv
Total rows: 712311

Columns containing missing values:
Std: 11
Variance: 11

Missing values in Std: 11
Missing values in Variance: 11

Labels associated with missing Std values:
{'DDOS-PSHACK_FLOOD': 1, 'DDOS-SYNONYMOUSIP_FLOOD': 1, 'DDOS-SYN_FLOOD': 1, 'MIRAI-UDPPLAIN': 1, 'MIRAI-GREIP_FLOOD': 1, 'BENIGN': 3, 'DDOS-ACK_FRAGMENTATION': 1, 'MIRAI-GREETH_FLOOD': 1, 'DDOS-ICMP_FRAGMENTATION': 1}

Labels associated with missing Variance values:
{'DDOS-PSHACK_FLOOD': 1, 'DDOS-SYNONYMOUSIP_FLOOD': 1, 'DDOS-SYN_FLOOD': 1, 'MIRAI-UDPPLAIN': 1, 'MIRAI-GREIP_FLOOD': 1, 'BENIGN': 3, 'DDOS-ACK_FRAGMENTATION': 1, 'MIRAI-GREETH_FLOOD': 1, 'DDOS-ICMP_FRAGMENTATION': 1}

Missing-value analysis: Merged02.csv
Total rows: 748585

Columns containing missing values:
Std: 6
Variance: 6

Missing values in Std: 6
Missing values in Variance: 6

Labels associated with missing Std values:
{'DDOS-PSHACK_FLOOD': 1, 'DDOS-ICMP_FLOOD': 1, 'DOS-UDP_FLOOD': 1, 'MIRAI-GREIP_FLOOD':

In [5]:
# TASK 2 - STEP 3
# Count positive and negative infinity values
# No data is modified or saved in this step.

import numpy as np

infinity_results = []

for filename in FILES:

    path = DATA_DIR / filename

    total_positive_inf = 0
    total_negative_inf = 0
    column_counts = {}

    for chunk in pd.read_csv(
        path,
        chunksize=CHUNK_SIZE,
        low_memory=False
    ):

        numeric = chunk.select_dtypes(include=[np.number])

        positive_inf = np.isposinf(numeric)
        negative_inf = np.isneginf(numeric)

        total_positive_inf += int(positive_inf.sum().sum())
        total_negative_inf += int(negative_inf.sum().sum())

        for col in numeric.columns:

            pos_count = int(positive_inf[col].sum())
            neg_count = int(negative_inf[col].sum())

            if pos_count or neg_count:

                if col not in column_counts:
                    column_counts[col] = {
                        "positive_inf": 0,
                        "negative_inf": 0
                    }

                column_counts[col]["positive_inf"] += pos_count
                column_counts[col]["negative_inf"] += neg_count

    print(f"\n{'=' * 50}")
    print(filename)
    print("Positive infinity:", total_positive_inf)
    print("Negative infinity:", total_negative_inf)
    print("Total infinity:", total_positive_inf + total_negative_inf)
    print("Affected columns:", column_counts)

    infinity_results.append({
        "file": filename,
        "positive_infinity": total_positive_inf,
        "negative_infinity": total_negative_inf,
        "total_infinity": total_positive_inf + total_negative_inf
    })


Merged01.csv
Positive infinity: 14
Negative infinity: 0
Total infinity: 14
Affected columns: {'Rate': {'positive_inf': 14, 'negative_inf': 0}}

Merged02.csv
Positive infinity: 13
Negative infinity: 0
Total infinity: 13
Affected columns: {'Rate': {'positive_inf': 13, 'negative_inf': 0}}


In [6]:
# TASK 2 - STEP 4
# Convert positive and negative infinity to NaN.
# Original CSV files remain unchanged.

import numpy as np
import pandas as pd

invalid_value_results = []

for filename in FILES:

    path = DATA_DIR / filename

    total_rows = 0
    before_missing = 0
    positive_inf_count = 0
    negative_inf_count = 0
    after_missing = 0
    affected_columns = {}

    for chunk in pd.read_csv(
        path,
        chunksize=CHUNK_SIZE,
        low_memory=False
    ):

        total_rows += len(chunk)

        # Count missing cells before conversion
        before_missing += int(chunk.isna().sum().sum())

        # Select numeric columns
        numeric = chunk.select_dtypes(include=[np.number])

        # Count infinities before conversion
        positive_inf_count += int(
            np.isposinf(numeric).sum().sum()
        )

        negative_inf_count += int(
            np.isneginf(numeric).sum().sum()
        )

        # Record columns containing infinity
        for col in numeric.columns:
            count = int(np.isinf(numeric[col]).sum())

            if count > 0:
                affected_columns[col] = (
                    affected_columns.get(col, 0) + count
                )

        # Convert positive and negative infinity to NaN
        chunk.replace(
            [np.inf, -np.inf],
            np.nan,
            inplace=True
        )

        # Count missing cells after conversion
        after_missing += int(chunk.isna().sum().sum())

    invalid_value_results.append({
        "file": filename,
        "rows_checked": total_rows,
        "missing_before": before_missing,
        "positive_infinity": positive_inf_count,
        "negative_infinity": negative_inf_count,
        "infinity_converted": (
            positive_inf_count + negative_inf_count
        ),
        "missing_after_conversion": after_missing
    })

    print(f"\n{'=' * 50}")
    print(filename)
    print("Rows checked:", total_rows)
    print("Missing before:", before_missing)
    print("Positive infinity:", positive_inf_count)
    print("Negative infinity:", negative_inf_count)
    print("Affected columns:", affected_columns)
    print("Missing after conversion:", after_missing)

print("\nOriginal files were not modified.")


Merged01.csv
Rows checked: 712311
Missing before: 22
Positive infinity: 14
Negative infinity: 0
Affected columns: {'Rate': 14}
Missing after conversion: 36

Merged02.csv
Rows checked: 748585
Missing before: 12
Positive infinity: 13
Negative infinity: 0
Affected columns: {'Rate': 13}
Missing after conversion: 25

Original files were not modified.


In [7]:
%pip install duckdb

   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
   - -------------------------------------- 0.5/13.2 MB 2.1 MB/s eta 0:00:07
   --- ------------------------------------ 1.0/13.2 MB 2.4 MB/s eta 0:00:06
   ---- ----------------------------------- 1.6/13.2 MB 2.4 MB/s eta 0:00:05
   ------ --------------------------------- 2.1/13.2 MB 2.4 MB/s eta 0:00:05
   -------- ------------------------------- 2.9/13.2 MB 2.8 MB/s eta 0:00:04
   ----------- ---------------------------- 3.7/13.2 MB 3.0 MB/s eta 0:00:04
   ------------- -------------------------- 4.5/13.2 MB 3.1 MB/s eta 0:00:03
   --------------- ------------------------ 5.2/13.2 MB 3.2 MB/s eta 0:00:03
   -------------------- ------------------- 6.8/13.2 MB 3.6 MB/s eta 0:00:02
   ------------------------ --------------- 8.1/13.2 MB 3.9 MB/s eta 0:00:02
   ----------------------------- ---------- 9.7/13.2 MB 4.3 MB/s eta 0:00:01
   ----------

In [26]:

# TASK 2 - STEP 5
# Corrected duplicate-record audit using DuckDB
# Original CSV files are not modified.

import duckdb
import pandas as pd

DB_PATH = TASK2_DIR / "duplicate_check.duckdb"
DUPLICATE_REPORT = REPORT_DIR / "duplicate_report.csv"

con = duckdb.connect(str(DB_PATH))

path1 = str(DATA_DIR / "Merged01.csv").replace("\\", "/")
path2 = str(DATA_DIR / "Merged02.csv").replace("\\", "/")

# Create temporary views of the original CSV files
con.execute(f"""
    CREATE OR REPLACE TEMP VIEW file1 AS
    SELECT * FROM read_csv(
        '{path1}',
        header = true,
        all_varchar = true
    )
""")

con.execute(f"""
    CREATE OR REPLACE TEMP VIEW file2 AS
    SELECT * FROM read_csv(
        '{path2}',
        header = true,
        all_varchar = true
    )
""")

duplicate_results = []

for filename, view_name in [
    ("Merged01.csv", "file1"),
    ("Merged02.csv", "file2")
]:
    # Count all rows
    total_rows = con.execute(
        f"SELECT COUNT(*) FROM {view_name}"
    ).fetchone()[0]

    # Count unique complete records
    distinct_rows = con.execute(
        f"""
        SELECT COUNT(*)
        FROM (
            SELECT DISTINCT *
            FROM {view_name}
        ) AS unique_records
        """
    ).fetchone()[0]

    # Count duplicate occurrences beyond the first copy
    duplicate_excess = total_rows - distinct_rows

    duplicate_results.append({
        "file": filename,
        "total_rows": total_rows,
        "distinct_complete_records": distinct_rows,
        "duplicate_excess_rows": duplicate_excess
    })

duplicate_report = pd.DataFrame(duplicate_results)

display(duplicate_report)

duplicate_report.to_csv(
    DUPLICATE_REPORT,
    index=False
)

print("Corrected duplicate report saved to:", DUPLICATE_REPORT)

con.close()

,file,total_rows,distinct_complete_records,duplicate_excess_rows
0,Merged01.csv,712311,544943,167368
1,Merged02.csv,748585,568678,179907


Corrected duplicate report saved to: C:\Users\panda\OneDrive\Desktop\GreenFed-IDS-Audit\TASK-2\reports\duplicate_report.csv


In [66]:
# TASK 2 - STEP 6
# Cross-file duplicate and label-conflict audit

import duckdb
import pandas as pd

DB_PATH = TASK2_DIR / "duplicate_check.duckdb"
OVERLAP_REPORT = REPORT_DIR / "cross_file_overlap.csv"

con = duckdb.connect(str(DB_PATH))
con.execute("SET enable_progress_bar = false")  # avoids the widget error

path1 = str(DATA_DIR / "Merged01.csv").replace("\\", "/")
path2 = str(DATA_DIR / "Merged02.csv").replace("\\", "/")

for name, path in [("file1", path1), ("file2", path2)]:
    con.execute(f"""
        CREATE OR REPLACE TEMP VIEW {name} AS
        SELECT * FROM read_csv('{path}', header=true, all_varchar=true)
    """)

cols1 = con.execute("DESCRIBE file1").fetchdf()["column_name"].tolist()
cols2 = con.execute("DESCRIBE file2").fetchdf()["column_name"].tolist()
if cols1 != cols2:
    raise ValueError("The two files have different column structures.")
columns = cols1

def q(column):
    return '"' + column.replace('"', '""') + '"'

# Count distinct complete records in each file
distinct1 = con.execute("""
    SELECT COUNT(*)
    FROM (SELECT DISTINCT * FROM file1)
""").fetchone()[0]

distinct2 = con.execute("""
    SELECT COUNT(*)
    FROM (SELECT DISTINCT * FROM file2)
""").fetchone()[0]

print("Distinct records in Merged01:", distinct1)
print("Distinct records in Merged02:", distinct2)

# Count exact complete-record matches across both files
# (NULL-safe equality so records with blanks can still match)
join_all = " AND ".join(
    f"a.{q(c)} IS NOT DISTINCT FROM b.{q(c)}" for c in columns
)

shared = con.execute(f"""
    SELECT COUNT(*)
    FROM (
        SELECT DISTINCT a.*
        FROM file1 a
        INNER JOIN file2 b
            ON {join_all}
    ) AS shared_records
""").fetchone()[0]

print("Shared distinct complete records:", shared)


# Verify feature-identical records with conflicting labels

feature_columns = [c for c in columns if c != "Label"]

quoted_features = ", ".join(
    f'"{c.replace(chr(34), chr(34) * 2)}"'
    for c in feature_columns
)

feature_group_query = f"""
    SELECT
        {quoted_features},
        COUNT(DISTINCT "Label") AS distinct_label_count
    FROM (
        SELECT * FROM file1
        UNION ALL
        SELECT * FROM file2
    ) AS combined
    GROUP BY {quoted_features}
    HAVING COUNT(DISTINCT "Label") > 1
"""

conflict_summary = con.execute(f"""
    SELECT
        COUNT(*) AS conflicting_feature_combinations
    FROM ({feature_group_query}) AS conflicts
""").fetchdf()

display(conflict_summary)

print(
    "Unique feature combinations with multiple labels:",
    int(conflict_summary.iloc[0, 0])
)


# Build the verified cross-file overlap report

# Update the overlap report with the verified metrics

verified_conflicts = int(
    conflict_summary["conflicting_feature_combinations"].iloc[0]
)

overlap_report = pd.DataFrame([{
    "distinct_records_merged01": distinct1,
    "distinct_records_merged02": distinct2,
    "shared_distinct_complete_records": shared,
    "feature_label_conflict_records": verified_conflicts
}])

display(overlap_report)

OVERLAP_REPORT = REPORT_DIR / "cross_file_overlap.csv"
overlap_report.to_csv(OVERLAP_REPORT, index=False)

print("Updated overlap report saved to:", OVERLAP_REPORT)


con.close()

Distinct records in Merged01: 544943
Distinct records in Merged02: 568678
Shared distinct complete records: 99430


,conflicting_feature_combinations
0,44540


Unique feature combinations with multiple labels: 44540


,distinct_records_merged01,distinct_records_merged02,shared_distinct_complete_records,feature_label_conflict_records
0,544943,568678,99430,44540


Updated overlap report saved to: C:\Users\panda\OneDrive\Desktop\GreenFed-IDS-Audit\TASK-2\reports\cross_file_overlap.csv


In [55]:
# TASK 2 - STEP 6
# Verify label distribution across the complete original files.

label_results = []

for filename in FILES:

    path = DATA_DIR / filename
    label_counts = {}
    total_rows = 0

    for chunk in pd.read_csv(
        path,
        chunksize=CHUNK_SIZE,
        low_memory=False
    ):

        total_rows += len(chunk)

        counts = chunk["Label"].value_counts(dropna=False)

        for label, count in counts.items():
            label_counts[label] = (
                label_counts.get(label, 0) + int(count)
            )

    print(f"\n{'=' * 60}")
    print(filename)
    print("Total rows:", total_rows)
    print("Distinct labels:", len(label_counts))

    print("\nLabel distribution (smallest counts first):")

    for label, count in sorted(
        label_counts.items(),
        key=lambda item: item[1]
    ):
        print(f"{label}: {count}")

    for label, count in label_counts.items():
        label_results.append({
            "file": filename,
            "Label": label,
            "count": count
        })

label_report = pd.DataFrame(label_results)

label_report.to_csv(
    REPORT_DIR / "label_distribution_before.csv",
    index=False
)

print("\nLabel report saved.")


Merged01.csv
Total rows: 712311
Distinct labels: 34

Label distribution (smallest counts first):
UPLOADING_ATTACK: 20
RECON-PINGSWEEP: 28
BACKDOOR_MALWARE: 41
XSS: 67
COMMANDINJECTION: 73
SQLINJECTION: 77
BROWSERHIJACKING: 83
DICTIONARYBRUTEFORCE: 204
DDOS-SLOWLORIS: 354
DDOS-HTTP_FLOOD: 416
DOS-HTTP_FLOOD: 1113
RECON-PORTSCAN: 1251
RECON-OSSCAN: 1433
RECON-HOSTDISCOVERY: 2045
DNS_SPOOFING: 2738
DDOS-UDP_FRAGMENTATION: 4264
DDOS-ACK_FRAGMENTATION: 4308
MITM-ARPSPOOFING: 4590
VULNERABILITYSCAN: 5805
DDOS-ICMP_FRAGMENTATION: 6784
MIRAI-GREIP_FLOOD: 11187
MIRAI-UDPPLAIN: 13342
MIRAI-GREETH_FLOOD: 15135
BENIGN: 16577
DOS-SYN_FLOOD: 30620
DOS-TCP_FLOOD: 40391
DOS-UDP_FLOOD: 50371
DDOS-SYNONYMOUSIP_FLOOD: 54749
DDOS-SYN_FLOOD: 61460
DDOS-RSTFINFLOOD: 61652
DDOS-PSHACK_FLOOD: 62171
DDOS-TCP_FLOOD: 68289
DDOS-UDP_FLOOD: 82011
DDOS-ICMP_FLOOD: 108662

Merged02.csv
Total rows: 748585
Distinct labels: 34

Label distribution (smallest counts first):
UPLOADING_ATTACK: 19
RECON-PINGSWEEP: 27
BACKDO

In [56]:
from pathlib import Path
import pandas as pd
import numpy as np

# Original CSV files
DATA_DIR = Path(
    r"C:\Users\panda\Downloads\MERGED_CSV\MERGED_CSV"
)

# Your actual Task 2 folder
TASK2_DIR = Path(
    r"C:\Users\panda\OneDrive\Desktop\GreenFed-IDS-Audit\TASK-2"
)

# Output and report folders
REPORT_DIR = TASK2_DIR / "reports"
OUTPUT_DIR = TASK2_DIR / "output"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FILES = ["Merged01.csv", "Merged02.csv"]
CHUNK_SIZE = 100_000

print("Task 2 folder:", TASK2_DIR)
print("Output folder:", OUTPUT_DIR)
print("Source folder exists:", DATA_DIR.exists())
print("Task 2 folder exists:", TASK2_DIR.exists())

Task 2 folder: C:\Users\panda\OneDrive\Desktop\GreenFed-IDS-Audit\TASK-2
Output folder: C:\Users\panda\OneDrive\Desktop\GreenFed-IDS-Audit\TASK-2\output
Source folder exists: True
Task 2 folder exists: True


In [57]:
import numpy as np
import pandas as pd

In [58]:
def calculate_medians(file_path):
    columns = ["Rate", "Std", "Variance"]
    valid_values = {col: [] for col in columns}

    for chunk in pd.read_csv(
        file_path,
        usecols=columns,
        chunksize=CHUNK_SIZE
    ):
        for col in columns:
            values = pd.to_numeric(chunk[col], errors="coerce")
            values = values[np.isfinite(values)]
            valid_values[col].append(values.to_numpy())

    medians = {}

    for col in columns:
        all_values = np.concatenate(valid_values[col])
        medians[col] = float(np.median(all_values))

    return medians

In [59]:
for filename in FILES:
    input_path = DATA_DIR / filename
    output_path = OUTPUT_DIR / filename.replace(".csv", "_clean.csv")

    print(f"\nProcessing: {filename}")

    # Calculate medians using the entire source file
    medians = calculate_medians(input_path)
    print("Median values:", medians)

    first_chunk = True

    # Process the source file in chunks
    for chunk in pd.read_csv(input_path, chunksize=CHUNK_SIZE):

        # Replace infinity with NaN in numeric columns
        numeric_columns = chunk.select_dtypes(include="number").columns
        chunk[numeric_columns] = chunk[numeric_columns].replace(
            [np.inf, -np.inf], np.nan
        )

        # Fill missing feature values using the calculated medians
        for col, median_value in medians.items():
            chunk[col] = chunk[col].fillna(median_value)

        # Write the cleaned chunk
        chunk.to_csv(
            output_path,
            mode="w" if first_chunk else "a",
            header=first_chunk,
            index=False
        )

        first_chunk = False

    print(f"Cleaned file saved: {output_path}")


Processing: Merged01.csv
Median values: {'Rate': 24613.015668094595, 'Std': 0.0, 'Variance': 0.0}
Cleaned file saved: C:\Users\panda\OneDrive\Desktop\GreenFed-IDS-Audit\TASK-2\output\Merged01_clean.csv

Processing: Merged02.csv
Median values: {'Rate': 24588.486340719897, 'Std': 0.0, 'Variance': 0.0}
Cleaned file saved: C:\Users\panda\OneDrive\Desktop\GreenFed-IDS-Audit\TASK-2\output\Merged02_clean.csv


In [60]:
import numpy as np
import pandas as pd

validation_results = []

for filename in FILES:
    original_path = DATA_DIR / filename
    clean_path = OUTPUT_DIR / filename.replace(".csv", "_clean.csv")

    print(f"\nValidating {filename}...")

    original_rows = 0
    original_columns = None
    original_labels = set()

    # Read the original file in chunks
    for chunk in pd.read_csv(original_path, chunksize=CHUNK_SIZE):
        original_rows += len(chunk)
        original_columns = list(chunk.columns)
        original_labels.update(chunk["Label"].dropna().unique())

    clean_rows = 0
    clean_columns = None
    missing_cells = 0
    infinity_cells = 0
    clean_labels = set()

    # Read the cleaned file in chunks
    for chunk in pd.read_csv(clean_path, chunksize=CHUNK_SIZE):
        clean_rows += len(chunk)
        clean_columns = list(chunk.columns)
        clean_labels.update(chunk["Label"].dropna().unique())

        missing_cells += int(chunk.isna().sum().sum())

        numeric_cols = chunk.select_dtypes(include="number").columns
        if len(numeric_cols) > 0:
            infinity_cells += int(
                np.isinf(chunk[numeric_cols].to_numpy()).sum()
            )

    result = {
        "file": filename,
        "original_rows": original_rows,
        "clean_rows": clean_rows,
        "rows_match": original_rows == clean_rows,
        "columns_match": original_columns == clean_columns,
        "missing_cells": missing_cells,
        "infinity_cells": infinity_cells,
        "labels_match": original_labels == clean_labels,
    }

    validation_results.append(result)
    print(result)

validation_df = pd.DataFrame(validation_results)

validation_path = REPORT_DIR / "cleaning_validation.csv"
validation_df.to_csv(validation_path, index=False)

print("\nValidation report saved to:", validation_path)
display(validation_df)


Validating Merged01.csv...
{'file': 'Merged01.csv', 'original_rows': 712311, 'clean_rows': 712311, 'rows_match': True, 'columns_match': True, 'missing_cells': 0, 'infinity_cells': 0, 'labels_match': True}

Validating Merged02.csv...
{'file': 'Merged02.csv', 'original_rows': 748585, 'clean_rows': 748585, 'rows_match': True, 'columns_match': True, 'missing_cells': 0, 'infinity_cells': 0, 'labels_match': True}

Validation report saved to: C:\Users\panda\OneDrive\Desktop\GreenFed-IDS-Audit\TASK-2\reports\cleaning_validation.csv


,file,original_rows,clean_rows,rows_match,columns_match,missing_cells,infinity_cells,labels_match
0,Merged01.csv,712311,712311,True,True,0,0,True
1,Merged02.csv,748585,748585,True,True,0,0,True


In [61]:
comparison_results = []

for filename in FILES:
    original_path = DATA_DIR / filename
    clean_path = OUTPUT_DIR / filename.replace(
        ".csv", "_clean.csv"
    )

    original_rows = 0
    original_missing = 0
    original_infinity = 0
    original_columns = None

    clean_rows = 0
    clean_missing = 0
    clean_infinity = 0
    clean_columns = None

    # Analyze original dataset
    for chunk in pd.read_csv(
        original_path, chunksize=CHUNK_SIZE
    ):
        original_rows += len(chunk)
        original_columns = len(chunk.columns)
        original_missing += int(chunk.isna().sum().sum())

        numeric_cols = chunk.select_dtypes(
            include="number"
        ).columns

        if len(numeric_cols) > 0:
            original_infinity += int(
                np.isinf(chunk[numeric_cols].to_numpy()).sum()
            )

    # Analyze cleaned dataset
    for chunk in pd.read_csv(
        clean_path, chunksize=CHUNK_SIZE
    ):
        clean_rows += len(chunk)
        clean_columns = len(chunk.columns)
        clean_missing += int(chunk.isna().sum().sum())

        numeric_cols = chunk.select_dtypes(
            include="number"
        ).columns

        if len(numeric_cols) > 0:
            clean_infinity += int(
                np.isinf(chunk[numeric_cols].to_numpy()).sum()
            )

    comparison_results.append({
        "file": filename,
        "original_rows": original_rows,
        "cleaned_rows": clean_rows,
        "original_columns": original_columns,
        "cleaned_columns": clean_columns,
        "original_missing_cells": original_missing,
        "cleaned_missing_cells": clean_missing,
        "original_infinity_cells": original_infinity,
        "cleaned_infinity_cells": clean_infinity,
        "rows_removed": original_rows - clean_rows
    })

comparison_df = pd.DataFrame(comparison_results)

comparison_path = REPORT_DIR / "before_after_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)

display(comparison_df)

print("Report saved to:", comparison_path)

,file,original_rows,cleaned_rows,original_columns,cleaned_columns,original_missing_cells,cleaned_missing_cells,original_infinity_cells,cleaned_infinity_cells,rows_removed
0,Merged01.csv,712311,712311,40,40,22,0,14,0,0
1,Merged02.csv,748585,748585,40,40,12,0,13,0,0


Report saved to: C:\Users\panda\OneDrive\Desktop\GreenFed-IDS-Audit\TASK-2\reports\before_after_comparison.csv


In [63]:
# Load the reports created in previous steps
validation_df = pd.read_csv(
    REPORT_DIR / "cleaning_validation.csv"
)

comparison_df = pd.read_csv(
    REPORT_DIR / "before_after_comparison.csv"
)

# Combine the reports
final_report = comparison_df.merge(
    validation_df[
        [
            "file",
            "rows_match",
            "columns_match",
            "labels_match"
        ]
    ],
    on="file",
    how="left"
)

# Add preprocessing status
final_report["preprocessing_status"] = final_report.apply(
    lambda row: (
        "PASS"
        if row["cleaned_missing_cells"] == 0
        and row["cleaned_infinity_cells"] == 0
        and row["rows_match"]
        and row["columns_match"]
        and row["labels_match"]
        and row["rows_removed"] == 0
        else "CHECK"
    ),
    axis=1
)

# Save final report
final_report_path = REPORT_DIR / "final_preprocessing_report.csv"
final_report.to_csv(final_report_path, index=False)

display(final_report)

print("Final report saved to:", final_report_path)

,file,original_rows,cleaned_rows,original_columns,cleaned_columns,original_missing_cells,cleaned_missing_cells,original_infinity_cells,cleaned_infinity_cells,rows_removed,rows_match,columns_match,labels_match,preprocessing_status
0,Merged01.csv,712311,712311,40,40,22,0,14,0,0,True,True,True,PASS
1,Merged02.csv,748585,748585,40,40,12,0,13,0,0,True,True,True,PASS


Final report saved to: C:\Users\panda\OneDrive\Desktop\GreenFed-IDS-Audit\TASK-2\reports\final_preprocessing_report.csv


In [47]:
import numpy as np
import pandas as pd

columns_to_check = ["Std", "Variance", "Rate"]

for filename in FILES:
    path = DATA_DIR / filename

    print(f"\n--- {filename} ---")

    values_by_column = {
        col: [] for col in columns_to_check
    }

    for chunk in pd.read_csv(
        path,
        usecols=columns_to_check,
        chunksize=CHUNK_SIZE
    ):
        for col in columns_to_check:
            values = pd.to_numeric(
                chunk[col], errors="coerce"
            )

            # Exclude infinity and missing values
            values = values[np.isfinite(values)]
            values_by_column[col].append(values.to_numpy())

    for col, arrays in values_by_column.items():
        values = np.concatenate(arrays)

        print(f"\nColumn: {col}")
        print("Valid finite values:", len(values))
        print("Zero values:", int(np.sum(values == 0)))
        print("Median:", float(np.median(values)))
        print("Minimum:", float(np.min(values)))
        print("Maximum:", float(np.max(values)))


--- Merged01.csv ---

Column: Std
Valid finite values: 712300
Zero values: 526130
Median: 0.0
Minimum: 0.0
Maximum: 9533.007817869217

Column: Variance
Valid finite values: 712300
Zero values: 526130
Median: 0.0
Minimum: 0.0
Maximum: 90878238.05555557

Column: Rate
Valid finite values: 712297
Zero values: 0
Median: 24613.015668094595
Minimum: 0.0013495213981639
Maximum: 5242880.0

--- Merged02.csv ---

Column: Std
Valid finite values: 748579
Zero values: 552085
Median: 0.0
Minimum: 0.0
Maximum: 9523.200739947328

Column: Variance
Valid finite values: 748579
Zero values: 552085
Median: 0.0
Minimum: 0.0
Maximum: 90691352.33333331

Column: Rate
Valid finite values: 748572
Zero values: 0
Median: 24588.486340719897
Minimum: 0.0003019865525312
Maximum: 7340032.0


In [48]:

# TASK 2 - STEP 7
# Verify label distributions before and after cleaning

import pandas as pd

label_results = []

for filename in FILES:
    original_path = DATA_DIR / filename
    clean_path = OUTPUT_DIR / filename.replace(
        ".csv", "_clean.csv"
    )

    original_counts = {}
    clean_counts = {}

    # Count labels in the original file
    for chunk in pd.read_csv(
        original_path,
        usecols=["Label"],
        chunksize=CHUNK_SIZE
    ):
        counts = chunk["Label"].value_counts(dropna=False)
        for label, count in counts.items():
            original_counts[label] = (
                original_counts.get(label, 0) + int(count)
            )

    # Count labels in the cleaned file
    for chunk in pd.read_csv(
        clean_path,
        usecols=["Label"],
        chunksize=CHUNK_SIZE
    ):
        counts = chunk["Label"].value_counts(dropna=False)
        for label, count in counts.items():
            clean_counts[label] = (
                clean_counts.get(label, 0) + int(count)
            )

    all_labels = set(original_counts) | set(clean_counts)

    for label in all_labels:
        before = original_counts.get(label, 0)
        after = clean_counts.get(label, 0)

        label_results.append({
            "file": filename,
            "Label": label,
            "before_count": before,
            "after_count": after,
            "difference": after - before,
            "matches": before == after
        })

label_comparison = pd.DataFrame(label_results)

label_report_path = REPORT_DIR / "label_distribution_comparison.csv"
label_comparison.to_csv(label_report_path, index=False)

display(label_comparison.sort_values(["file", "before_count"]))

print("Total label rows checked:", len(label_comparison))
print("All label counts match:", label_comparison["matches"].all())
print("Report saved to:", label_report_path)


,file,Label,before_count,after_count,difference,matches
14,Merged01.csv,UPLOADING_ATTACK,20,20,0,True
7,Merged01.csv,RECON-PINGSWEEP,28,28,0,True
19,Merged01.csv,BACKDOOR_MALWARE,41,41,0,True
15,Merged01.csv,XSS,67,67,0,True
5,Merged01.csv,COMMANDINJECTION,73,73,0,True
...,...,...,...,...,...,...
46,Merged02.csv,DDOS-SYN_FLOOD,64824,64824,0,True
42,Merged02.csv,DDOS-PSHACK_FLOOD,65513,65513,0,True
35,Merged02.csv,DDOS-TCP_FLOOD,71210,71210,0,True
45,Merged02.csv,DDOS-UDP_FLOOD,85995,85995,0,True


Total label rows checked: 68
All label counts match: True
Report saved to: C:\Users\panda\OneDrive\Desktop\GreenFed-IDS-Audit\TASK-2\reports\label_distribution_comparison.csv


In [64]:

# TASK 2 - FINAL REVIEW SUMMARY

import pandas as pd

duplicate_df = pd.read_csv(
    REPORT_DIR / "duplicate_report.csv"
)

overlap_df = pd.read_csv(
    REPORT_DIR / "cross_file_overlap.csv"
)

labels_df = pd.read_csv(
    REPORT_DIR / "label_distribution_comparison.csv"
)

validation_df = pd.read_csv(
    REPORT_DIR / "cleaning_validation.csv"
)

# Summarize duplicate counts
duplicate_df["audit_type"] = "within_file_duplicates"

# Prepare a separate overlap summary
overlap_summary = overlap_df.copy()
overlap_summary["audit_type"] = "cross_file_overlap"

# Check label preservation
label_summary = pd.DataFrame([{
    "audit_type": "label_distribution_check",
    "comparisons_checked": len(labels_df),
    "all_label_counts_match": bool(labels_df["matches"].all())
}])

# Check cleaning validation
validation_summary = pd.DataFrame([{
    "audit_type": "cleaning_validation",
    "all_rows_match": bool(validation_df["rows_match"].all()),
    "all_columns_match": bool(validation_df["columns_match"].all()),
    "all_labels_match": bool(validation_df["labels_match"].all()),
    "no_missing_values": bool((validation_df["missing_cells"] == 0).all()),
    "no_infinity_values": bool((validation_df["infinity_cells"] == 0).all())
}])

# Save a compact summary of the audit results
review_summary_path = REPORT_DIR / "mentor_review_summary.csv"

review_summary = pd.DataFrame([
    {
        "audit_type": "within_file_duplicates",
        "file": row["file"],
        "total_rows": row["total_rows"],
        "distinct_records": row["distinct_complete_records"],
        "duplicate_excess_rows": row["duplicate_excess_rows"]
    }
    for _, row in duplicate_df.iterrows()
])

review_summary.to_csv(review_summary_path, index=False)

display(review_summary)
display(overlap_summary)
display(label_summary)
display(validation_summary)

print("Mentor review summary saved to:", review_summary_path)


,audit_type,file,total_rows,distinct_records,duplicate_excess_rows
0,within_file_duplicates,Merged01.csv,712311,544943,167368
1,within_file_duplicates,Merged02.csv,748585,568678,179907


,distinct_records_merged01,distinct_records_merged02,shared_distinct_complete_records,feature_label_conflict_records,audit_type
0,544943,568678,99430,44540,cross_file_overlap


,audit_type,comparisons_checked,all_label_counts_match
0,label_distribution_check,68,True


,audit_type,all_rows_match,all_columns_match,all_labels_match,no_missing_values,no_infinity_values
0,cleaning_validation,True,True,True,True,True


Mentor review summary saved to: C:\Users\panda\OneDrive\Desktop\GreenFed-IDS-Audit\TASK-2\reports\mentor_review_summary.csv


In [49]:

# TASK 2 - STEP 8
# Build an updated final preprocessing summary

import pandas as pd

comparison_df = pd.read_csv(
    REPORT_DIR / "before_after_comparison.csv"
)

validation_df = pd.read_csv(
    REPORT_DIR / "cleaning_validation.csv"
)

duplicate_df = pd.read_csv(
    REPORT_DIR / "duplicate_report.csv"
)

overlap_df = pd.read_csv(
    REPORT_DIR / "cross_file_overlap.csv"
)

label_df = pd.read_csv(
    REPORT_DIR / "label_distribution_comparison.csv"
)

# Confirm every label count matches
labels_match = bool(label_df["matches"].all())

# Confirm every dataset passed validation
validation_passed = bool(
    validation_df["rows_match"].all()
    and validation_df["columns_match"].all()
    and validation_df["labels_match"].all()
    and (validation_df["missing_cells"] == 0).all()
    and (validation_df["infinity_cells"] == 0).all()
)

# Combine before/after results with validation
final_report = comparison_df.merge(
    validation_df[
        ["file", "rows_match", "columns_match", "labels_match"]
    ],
    on="file",
    how="left"
)

final_report["all_label_counts_match"] = labels_match
final_report["validation_passed"] = validation_passed

final_report_path = REPORT_DIR / "final_preprocessing_report.csv"
final_report.to_csv(final_report_path, index=False)

display(final_report)

print("Label distributions match:", labels_match)
print("Validation passed:", validation_passed)
print("Duplicate audit saved:", (REPORT_DIR / "duplicate_report.csv").exists())
print("Overlap audit saved:", (REPORT_DIR / "cross_file_overlap.csv").exists())
print("Updated final report saved to:", final_report_path)


,file,original_rows,cleaned_rows,original_columns,cleaned_columns,original_missing_cells,cleaned_missing_cells,original_infinity_cells,cleaned_infinity_cells,rows_removed,rows_match,columns_match,labels_match,all_label_counts_match,validation_passed
0,Merged01.csv,712311,712311,40,40,22,0,14,0,0,True,True,True,True,True
1,Merged02.csv,748585,748585,40,40,12,0,13,0,0,True,True,True,True,True


Label distributions match: True
Validation passed: True
Duplicate audit saved: True
Overlap audit saved: True
Updated final report saved to: C:\Users\panda\OneDrive\Desktop\GreenFed-IDS-Audit\TASK-2\reports\final_preprocessing_report.csv


In [51]:

readme_content = """
# Task 2: Dataset Preprocessing

## Objective
Clean and validate Merged01.csv and Merged02.csv.

## Input Datasets
- Merged01.csv: 712,311 rows and 40 columns
- Merged02.csv: 748,585 rows and 40 columns

Each dataset contains 39 features and one target column: Label.

## Preprocessing Performed
1. Verified dataset structure and column consistency.
2. Investigated missing values and infinity values.
3. Replaced positive and negative infinity with missing values.
4. Imputed missing values in Rate, Std, and Variance using
   the median calculated separately for each file.
5. Preserved all rows, labels, columns, and duplicate records.
6. Validated the cleaned datasets.
7. Verified that all 34 label counts match before and after cleaning.

## Results

| Dataset | Original Rows | Cleaned Rows | Original Missing Cells | Original Infinity Values |
|---|---:|---:|---:|---:|
| Merged01.csv | 712311 | 712311 | 22 | 14 |
| Merged02.csv | 748585 | 748585 | 12 | 13 |

Both cleaned datasets contain zero missing cells and zero
infinity values according to the validation report.

## Duplicate Audit

| Dataset | Total Rows | Distinct Records | Duplicate Excess Rows |
|---|---:|---:|---:|
| Merged01.csv | 712311 | 544943 | 167368 |
| Merged02.csv | 748585 | 568678 | 179907 |

Duplicate records were retained.

## Cross-File Overlap
The audit identified 99,430 distinct complete records shared
between the two files. A further investigation of feature-identical
records with conflicting labels is required before finalizing
the evaluation strategy.

## Output Files
- output/Merged01_clean.csv
- output/Merged02_clean.csv

## Reports
- reports/before_after_comparison.csv
- reports/cleaning_validation.csv
- reports/cross_file_overlap.csv
- reports/duplicate_report.csv
- reports/final_preprocessing_report.csv
- reports/label_distribution_comparison.csv

## Scope
Only Merged01.csv and Merged02.csv were processed.
Duplicate records were not deleted, and labels were not changed.
No model training or train/test split was performed.

## Conclusion
Both datasets passed the current preprocessing validation checks.
All original rows and columns were preserved.
"""

readme_path = TASK2_DIR / "Readme.md"
readme_path.write_text(readme_content.strip(), encoding="utf-8")

print("README updated:", readme_path)
print("File exists:", readme_path.exists())


README updated: C:\Users\panda\OneDrive\Desktop\GreenFed-IDS-Audit\TASK-2\Readme.md
File exists: True


In [67]:

from pathlib import Path
import pandas as pd

# 1. Check required report files
required_files = [
    "before_after_comparison.csv",
    "cleaning_validation.csv",
    "cross_file_overlap.csv",
    "duplicate_report.csv",
    "final_preprocessing_report.csv",
    "label_distribution_comparison.csv",
]

file_check = pd.DataFrame([
    {
        "file": name,
        "exists": (REPORT_DIR / name).exists()
    }
    for name in required_files
])

display(file_check)

# 2. Verify label counts
labels = pd.read_csv(
    REPORT_DIR / "label_distribution_comparison.csv"
)

print("Label comparisons:", len(labels))
print("All label counts match:", labels["matches"].all())

# 3. Verify duplicate counts
duplicates = pd.read_csv(
    REPORT_DIR / "duplicate_report.csv"
)

display(duplicates)

# 4. Verify cross-file overlap
overlap = pd.read_csv(
    REPORT_DIR / "cross_file_overlap.csv"
)

display(overlap)

# 5. Verify cleaned datasets
validation = pd.read_csv(
    REPORT_DIR / "cleaning_validation.csv"
)

display(validation)

print(
    "All required reports exist:",
    file_check["exists"].all()
)


,file,exists
0,before_after_comparison.csv,True
1,cleaning_validation.csv,True
2,cross_file_overlap.csv,True
3,duplicate_report.csv,True
4,final_preprocessing_report.csv,True
5,label_distribution_comparison.csv,True


Label comparisons: 68
All label counts match: True


,file,total_rows,distinct_complete_records,duplicate_excess_rows
0,Merged01.csv,712311,544943,167368
1,Merged02.csv,748585,568678,179907


,distinct_records_merged01,distinct_records_merged02,shared_distinct_complete_records,feature_label_conflict_records
0,544943,568678,99430,44540


,file,original_rows,clean_rows,rows_match,columns_match,missing_cells,infinity_cells,labels_match
0,Merged01.csv,712311,712311,True,True,0,0,True
1,Merged02.csv,748585,748585,True,True,0,0,True


All required reports exist: True


In [65]:
from pathlib import Path

readme_path = Path("Readme.md")

if readme_path.exists():
    print("README already exists.")
    print("It will not be overwritten.")
else:
    print("README not found. No file was changed.")

README already exists.
It will not be overwritten.
